In [1]:
# !pip install transformers datasets

In [ ]:
import tensorflow as tf
from transformers import TFBertForSequenceClassification,\
BertTokenizer, create_optimizer
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

In [3]:
import pandas as pd
data=pd.read_csv('/content/IMDB Dataset.csv')
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
data['sentiment']=data['sentiment'].map({'positive':1,'negative':0})

In [5]:
data

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [6]:
import re
def clean_review(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,!?\'\" ]', '', text)
    text = text.strip()
    return text

In [7]:
data['review'] = data['review'].apply(clean_review)

In [8]:
from datasets import Dataset
dataset = Dataset.from_pandas(data)
dataset

Dataset({
    features: ['review', 'sentiment'],
    num_rows: 50000
})

In [9]:
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [10]:
# Tokenize the text inputs
def tokenize_function(example):
    return tokenizer(example["review"], padding="max_length", truncation=True)

In [11]:
# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
tokenized_datasets

Dataset({
    features: ['review', 'sentiment', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50000
})

In [13]:
len(tokenized_datasets['review'])

50000

In [14]:
len(tokenized_datasets['input_ids'][1])

512

In [15]:
len(tokenized_datasets['token_type_ids'][1])

512

In [16]:
tokenized_datasets['attention_mask'][1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [17]:
len(tokenized_datasets['attention_mask'][1])

512

In [18]:
len(tokenized_datasets['input_ids'][0])
len(tokenized_datasets['input_ids'][1])

512

In [19]:
tokenized_datasets

Dataset({
    features: ['review', 'sentiment', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50000
})

In [20]:
split_datasets = tokenized_datasets.train_test_split(test_size=0.2)

In [21]:
split_datasets

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['review', 'sentiment', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [22]:
split_datasets["train"].to_tf_dataset() # All the columns
split_datasets["train"].to_tf_dataset(columns=["input_ids", "attention_mask"]) # Input columns selected
split_datasets["train"].to_tf_dataset(columns=["input_ids", "attention_mask"],
                                     label_cols=["sentiment"]) # Input and label columns selected

/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py:403: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


<_PrefetchDataset element_spec=({'input_ids': TensorSpec(shape=(512,), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(512,), dtype=tf.int64, name=None)}, TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [23]:
train_dataset = split_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],  # Input features
    label_cols=["sentiment"],                # Label column
    shuffle=True,                            # Shuffle for training
    batch_size=8,                           # Batch size for training
)

In [24]:
train_dataset

<_PrefetchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 512), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, 512), dtype=tf.int64, name=None)}, TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [25]:
for batch in train_dataset.take(1):  # Take 1 batch
    print(batch)


({'input_ids': <tf.Tensor: shape=(8, 512), dtype=int64, numpy=
array([[  101,  1045,  2031, ...,     0,     0,     0],
       [  101,  2339,  2107, ...,     0,     0,     0],
       [  101,  1045,  2031, ...,     0,     0,     0],
       ...,
       [  101,  3432,  2028, ...,     0,     0,     0],
       [  101, 16432,  2742, ...,     0,     0,     0],
       [  101,  1996,  2204, ...,  2009,  1005,   102]])>, 'attention_mask': <tf.Tensor: shape=(8, 512), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1]])>}, <tf.Tensor: shape=(8,), dtype=int64, numpy=array([0, 0, 0, 0, 0, 1, 1, 0])>)


In [26]:
num_batches = train_dataset.cardinality().numpy()
print(f"Number of batches: {num_batches}")


Number of batches: 5000


In [27]:
eval_dataset = split_datasets["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],  # Input features
    label_cols=["sentiment"],                # Label column
    shuffle=False,                           # No shuffle for evaluation
    batch_size=8,                           # Batch size for evaluation
)

In [28]:
train_dataset.cardinality().numpy()
eval_dataset.cardinality().numpy()

np.int64(1250)

In [29]:
num_train_batches = train_dataset.cardinality().numpy()
num_eval_batches = eval_dataset.cardinality().numpy()
print(f"Number of training batches: {num_train_batches}")
print(f"Number of evaluation batches: {num_eval_batches}")

Number of training batches: 5000
Number of evaluation batches: 1250


In [30]:
!pip install --upgrade transformers tensorflow safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.6 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is inc

In [31]:
from transformers import TFDistilBertForSequenceClassification
model = TFDistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    from_pt=True,
    num_labels=2
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_projector.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSeq

In [32]:
model.summary()

Model: "tf_distil_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
 dropout_19 (Dropout)        multiple                  0         
                                                                 
Total params: 66955010 (255.41 MB)
Trainable params: 66955010 (255.41 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [33]:
# Define optimizer, loss, and metrics
num_epochs = 2
steps_per_epoch = len(train_dataset) # 891 steps
num_train_steps = steps_per_epoch * num_epochs # 891*2
optimizer, schedule = create_optimizer(init_lr=2e-5,  # ADAM
                                       num_warmup_steps=0,
                                       num_train_steps=num_train_steps)

In [34]:
optimizer,schedule

(<tf_keras.src.optimizers.adam.Adam at 0x7bb51847fdd0>,
 <tf_keras.src.optimizers.schedules.learning_rate_schedule.PolynomialDecay at 0x7bb518420530>)

In [35]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]

In [36]:
model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)

In [37]:
# Train the model
history=model.fit(train_dataset,
                  validation_data=eval_dataset,
                  epochs=num_epochs,
                  verbose=True)

Epoch 1/2
5000/5000 [==============================] - 2620s 517ms/step - loss: 0.2250 - sparse_categorical_accuracy: 0.9102 - val_loss: 0.1745 - val_sparse_categorical_accuracy: 0.9322
Epoch 2/2
5000/5000 [==============================] - 2588s 518ms/step - loss: 0.1019 - sparse_categorical_accuracy: 0.9644 - val_loss: 0.1798 - val_sparse_categorical_accuracy: 0.9360


In [39]:
# Evaluate the model
eval_results = model.evaluate(eval_dataset)
print(f"Evaluation results: {eval_results}")

# Get predictions
predictions = model.predict(eval_dataset)
logits = predictions['logits']
pred_labels = tf.argmax(logits, axis=-1).numpy()

# Get true labels
# Extract true labels from the dataset and flatten the list of tensors
true_labels = tf.concat([y for x, y in eval_dataset], axis=0).numpy()

# Compute metrics
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')
accuracy = accuracy_score(true_labels, pred_labels)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

1250/1250 [==============================] - 208s 167ms/step - loss: 0.1798 - sparse_categorical_accuracy: 0.9360
Evaluation results: [0.17982517182826996, 0.9359999895095825]
1250/1250 [==============================] - 206s 165ms/step
Accuracy: 0.936
Precision: 0.9386138613861386
Recall: 0.9349112426035503
F1 Score: 0.9367588932806324


In [40]:
# Save the model
model.save('checkpoints/my_bert_model')

In [41]:
# Load the model
loaded_model = tf.keras.models.load_model('checkpoints/my_bert_model')

In [42]:
from transformers import BertTokenizer
import numpy as np

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Define a test sample
test_sample = "I love this product! It's amazing."

# Tokenize the test sample
inputs = tokenizer(test_sample, return_tensors='tf', padding='max_length', truncation=True, max_length=128)

# Print the tokenized inputs for verification
print("Tokenized Inputs:", inputs)


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Tokenized Inputs: {'input_ids': <tf.Tensor: shape=(1, 128), dtype=int32, numpy=
array([[ 101, 1045, 2293, 2023, 4031,  999, 2009, 1005, 1055, 6429, 1012,
         102,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(1,

In [46]:
# Make predictions
predictions = loaded_model({'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask']})

# Print the keys of the predictions dictionary
print("Prediction keys:", predictions.keys())

# Extract logits and compute predicted class
logits = predictions['logits']
predicted_class = np.argmax(logits.numpy(), axis=-1)

# Print the prediction
print("Predicted Class:", predicted_class)

Prediction keys: dict_keys(['logits'])
Predicted Class: [1]
